# RadCluster_2_1 — Digital-Twin Campaign Control (T3 rev 6)

**Run this notebook unchanged on every participating machine. Nothing in it
is machine-specific and nothing needs editing.**

The host identifies itself from `machines.json`, which is also the single
source of the frozen grid — the notebook *builds its command line from that
file* rather than carrying its own copy, so the grid cannot drift between
machines and is never retyped. `git pull` in section 1 is the only way
settings change.

Run sections in order. **0–4 are pre-flight and must all pass before 5
launches the real run.**


## 0 — Setup


In [1]:
import json, subprocess, sys, time, collections
from pathlib import Path

def _find_root():
    """Locate digital_twin from wherever the kernel happens to start."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base, base / 'RadCluster_2_1' / 'digital_twin'):
            if (cand / 'run_ensemble.py').exists() and (cand / 'machines.json').exists():
                return cand
    raise SystemExit('cannot locate RadCluster_2_1/digital_twin from ' + str(Path.cwd()))

HERE = _find_root()
sys.path.insert(0, str(HERE))
import campaign_ops as ops, run_ensemble as RE

REPO     = Path(subprocess.run(['git','rev-parse','--show-toplevel'], cwd=HERE,
                               capture_output=True, text=True).stdout.strip())
REGISTRY = HERE / 'machines.json'
RESULTS  = HERE / 'results'
PY       = sys.executable
print('repo :', REPO)
print('here :', HERE)


repo : D:\Repos\RadCluster
here : D:\Repos\RadCluster\RadCluster_2_1\digital_twin


## 1 — Pull

Every participant must run the same code, design and grid. `merge_and_sobol`
compares `git_sha`, `solver_sha256`, `workbook_sha256`, `design_sha256`,
`run_cfg_sha`, `weights_sha` and `of`, and reports a PROVENANCE SPLIT on any
disagreement. **Re-run this cell whenever settings change — that is the only
step needed to pick up a new grid.**


In [2]:
print(subprocess.run(['git','pull','--ff-only'], cwd=REPO,
                     capture_output=True, text=True).stdout.strip())
print('HEAD =', subprocess.run(['git','rev-parse','--short','HEAD'], cwd=REPO,
                               capture_output=True, text=True).stdout.strip())


Already up to date.
HEAD = 739a160


## 2 — Read the frozen campaign settings

Everything below is derived from `machines.json`. Nothing is hard-coded here.


In [2]:
reg    = json.loads(REGISTRY.read_text())
G      = reg['grid']
DESIGN = HERE / reg['design']
EXPECTED_SHA = reg['run_cfg_sha']
TIMEOUT_S    = reg['timeout_s']

GRID = ['--equations', G['equations'],
        '--I', str(G['I']), '--V', str(G['V']),
        '--i-discrete', str(G['i_discrete']), '--v-discrete', str(G['v_discrete']),
        '--i-bin', str(G['i_bin']), '--v-bin', str(G['v_bin']),
        '--shape-function', G['shape_function'],
        '--i-mobile-default', str(G['i_mobile_default']),
        '--v-mobile-default', str(G['v_mobile_default']),
        '--dose', str(G['dose']), '--rtol', str(G['rtol']),
        '--solver-mode', G['solver_mode']]

print('design      ', DESIGN.name)
print('grid        ', ' '.join(f'{k}={v}' for k, v in G.items()))
print('run_cfg_sha ', EXPECTED_SHA)
print('timeout_s   ', TIMEOUT_S)
print('\nNOTE: i_mobile is PHYSICS, not a numerical knob. These rows cannot be')
print('pooled with any run at a different i_mobile; run_cfg_sha enforces it.')


design       T3_rev6.csv
grid         I=10000 V=5000 dose=1.0 equations=bin_moment solver_mode=active_window rtol=1e-06 i_discrete=40 v_discrete=5 i_bin=20 v_bin=20 shape_function=linear i_mobile_default=40 v_mobile_default=5
run_cfg_sha  482d59556632dff8
timeout_s    3600

NOTE: i_mobile is PHYSICS, not a numerical knob. These rows cannot be
pooled with any run at a different i_mobile; run_cfg_sha enforces it.


## 3 — Who am I?

Detection is by host fingerprint and **fails loudly** on no match or an
ambiguous match. A wrong index means two machines compute the same rows and
some rows are computed by nobody — which surfaces at merge time as *rows
MISSING*, indistinguishable from a machine that never reported.

If it refuses: `RE.register_host(<index>, REGISTRY)`, then commit and push
`machines.json`. Do not guess an index.

### Per-machine capacity and budget (added 2026-08-07)

Two registry fields are now allowed to differ per host. Neither can
re-partition the design:

| field | what it sets | scope |
|---|---|---|
| `slots` | worker processes (`run_ensemble` line ~1013) | local |
| `timeout_s` | per-row budget, overrides the global | local |
| `weight` | **the row → machine map** (line ~1044) | **campaign-wide, frozen** |

`slots` and `weight` are read independently, so capacity can be corrected
mid-campaign without a single completed row changing owner. `weights_map_sha`
is the hash to check: it must not move. **`weight` remains frozen.**

**Matrix-PC (index 1) was corrected on 2026-08-07** after it produced 52 rows
of which *all 52 were starved*, at a median 0.0027 dpa against a 1.0 dpa
target — reaching no rung of the dose ladder while still being marked
admissible. Two causes, both fixed here:

* **`slots` 4 → 20.** The host is 2× Xeon Gold 6230 (40 physical / 80 logical
  cores) and was provisioned as if it had four. It is *not* raised to 40:
  Windows splits >64 logical CPUs into two processor groups of 40, a process
  stays in the group it was created in, and only 20 **physical** cores are
  therefore reachable. Measured at 20 workers: each solver holds ~90 % of a
  core. At 36+ they would land on HT siblings and every row would get *longer*
  — the wrong trade when the binding constraint is a per-row deadline.
* **`timeout_s` 3600 → 50400.** The global 3600 s was calibrated from machine
  0's measured distribution — an M3 Max. At equal dose this host is ~13× slower
  (archived T2 campaign, both at median 0.1 dpa: 6990 s here vs 528 s on the
  Mac). Asking it for 1.0 dpa in 3600 s was roughly half the time it needed for
  a *tenth* of the dose, so every row was guaranteed to starve.

A host with no `timeout_s` key keeps the global value, so no other participant
changes behaviour. The notebook itself stays machine-agnostic: the value is
read from the registry like everything else.


In [3]:
me    = RE.detect_machine(reg, RE._host_facts())   # raises if unrecognised
W     = [float(w) for w in reg['weights'].split(',')]
rows, meta = RE.read_design(DESIGN)
mine  = [r for r in rows if RE.assign_machine(int(r['row_id']), reg['of'], W) == me['index']]
print(f"machine {me['index']} = {me['name']}   slots {me['slots']}   "
      f"speed {me['speed']} ({me['speed_source']})   weight {me['weight']}")
print(f'owns {len(mine)} of {len(rows)} rows')

# PER-MACHINE TIMEOUT OVERRIDE.  The global reg['timeout_s'] was calibrated
# from machine 0's MEASURED row-cost distribution (an M3 Max).  A host that is
# materially slower needs its own budget or it starves EVERY row and reaches no
# rung of the dose ladder -- which is worse than useless, because the rows are
# still marked admissible and still land in the pool.  A machine entry may
# therefore carry its own `timeout_s`; absent the key, the global value stands,
# so no other participant changes behaviour.  This is still not machine-specific
# NOTEBOOK code: the value is read from the registry like everything else.
TIMEOUT_S = float(me.get('timeout_s', reg['timeout_s']))
if 'timeout_s' in me:
    print(f"\ntimeout_s   {TIMEOUT_S:.0f} s  <- PER-MACHINE OVERRIDE for "
          f"{me['name']} (global is {float(reg['timeout_s']):.0f} s)")
else:
    print(f'\ntimeout_s   {TIMEOUT_S:.0f} s  (global; no override for this host)')


machine 1 = Matrix-PC   slots 20   speed 0.4 (DECLARED)   weight 1.6
owns 55 of 1008 rows

timeout_s   50400 s  <- PER-MACHINE OVERRIDE for Matrix-PC (global is 3600 s)


## 4 — Build, agreement gate, and a bounded PRE-FLIGHT

**Do not skip the pre-flight.** Its absence cost two aborted Hoffman2
submissions: every row failed in 5–8 s with `d100=nan` while the provenance
line looked perfect — that line prints *before* any row runs and proves only
that the config was assembled.

The cause is unresolved and **may affect any Linux host**: rows succeed
through a direct call but fail through `run_ensemble`'s multiprocessing pool
at the production grid. So it must run *on this machine*.


In [5]:
info = ops.ensure_solver()
r = subprocess.run([PY, str(HERE / 'check_machine.py')], cwd=HERE,
                   capture_output=True, text=True)
print(r.stdout[-1800:])
assert r.returncode == 0, 'AGREEMENT GATE FAILED — do not contribute rows from this build'


  solver: OK  D:\GitHub\RadCluster\RadCluster_2_1\build\Release\solver.exe  sha c28893e7d4a119e2


machine   Nasr-Workstation  (Windows-11-10.0.26200-SP0)
python    3.14.3
git       739a160c4ab0
solver    c28893e7d4a119e2  exists=True
workbook  9253e7a0370af966

running probe (I=150, 0.02 dpa, ~30 s) ...

  reference generated on Nasr-Workstation (git 76efc2a46f6c, solver c28893e7d4a119e2)
  note: git SHA differs from the reference machine - pull first if that is not intentional.

  field                    this machine        reference    rel diff
  Di_eff                7.085348836e-12  7.085348836e-12    0.00e+00
  Dv_eff                2.125201773e-13  2.125201773e-13    0.00e+00
  conv_psuccess         2.181172484e-06  2.181172484e-06    0.00e+00
  conv_psuccess_abs     1.000000000e+00  1.000000000e+00    0.00e+00
  N_loops_100           1.679711678e+20  1.679711678e+20    0.00e+00
  N_loops_111           2.904994914e+23  2.904994914e+23    0.00e+00
  mean_n_100            2.016423189e+02  2.016423189e+02    0.00e+00
  mean_n_111            3.604016025e+01  3.604016025e+01    0

In [6]:
import re as _re
t0 = time.time()
pf = RESULTS / '_preflight.jsonl'
r  = subprocess.run([PY, '-u', str(HERE / 'run_ensemble.py'),
                     '--design', str(DESIGN), '--machine', 'auto', *GRID,
                     '--timeout-s', str(TIMEOUT_S),
                     '--limit', '2', '--workers', '2', '--out', str(pf)],
                    cwd=HERE, capture_output=True, text=True)
out = r.stdout + r.stderr
print(out[-1800:])
m   = _re.search(r'"run_cfg_sha": "([0-9a-f]+)"', out)
sha = m.group(1) if m else None
print(f'\n  run_cfg_sha {sha}  (expected {EXPECTED_SHA})')
print(f'  FAIL lines  {out.count("FAIL")}')
print(f'  elapsed     {time.time()-t0:.0f} s   -> per-row cost sets the ETA')
assert sha == EXPECTED_SHA, 'GRID MISMATCH — this machine is not on the frozen grid'
assert out.count('FAIL') == 0, 'ROWS FAILED — do NOT launch; report the error text'
for p in RESULTS.glob('_preflight*'): p.unlink()
print('\n  PRE-FLIGHT PASSED')


  detected machine 3 = MacBook Air (DECLARED speed 0.85), 8 worker(s)
  weights from machines.json (sha e8b5c6f6dbcc5a75): 4 participants
machine 3/4  rows 2 (0 done, 2 to run)  workers 8
  provenance {"git_sha": "762c969c16bc6dbed4c7100a5ee2c3f38de587e4", "machine_id": "Mac.san.rr.com", "solver_sha256": "9366e57649416372", "workbook_sha256": "9253e7a0370af966", "design_sha256": "3461603bdf7818e3", "run_cfg_sha": "482d59556632dff8", "timeout_s": 12000.0, "stop_after_s": 0.0, "workers": 8, "weights_sha": "e8b5c6f6dbcc5a75", "of": 4, "equations": "bin_moment", "python": "3.9.6"}
  row      6   N2 AB  ok   d100= 22.85 N100=3.383e+20 pile=0.9999999299305912 dFP= 2.1e-01 4274s
  row      2   N2 AB  ok   d100= 22.85 N100=3.588e+20 pile=0.9999999393688225 dFP= 2.2e-01 4910s

done in 4910s: 2 admissible, 0 inadmissible, 0 failed -> /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/digital_twin/results/_preflight.jsonl
  manifest -> _preflight.manifest.json


  run_cfg_sha 482d59556632dff

## 5 — Launch

Detached, so the notebook can be closed — on **both** platforms now. That was a
POSIX-only promise until 2026-08-07: `start_new_session` is ignored on Windows,
and neither `DETACHED_PROCESS` nor `CREATE_BREAKAWAY_FROM_JOB` helps, because a
kernel is killed with `taskkill /F /T` and `/T` walks the parent-child table.
Windows therefore spawns through a throwaway launcher that exits immediately and
orphans the run. Measured: before this, a launch from the notebook died the
instant the kernel shut down — silently, after printing a healthy provenance
line.

Resumption is the design: re-running this cell skips `row_id`s already present
and only ever adds.

> `--timeout-s` is a budget, not a cap. On expiry the solver is asked to
> finalize and its **partial trajectory is kept**, contributing to every rung of
> the dose ladder it reached.
>
> It is a **ROW-level** budget. It used to be handed to every segment unchanged,
> so a row could run `n_segments` × the limit — rows of 16 108–20 503 s completed
> under a 12 000 s setting and the cap never bit. Fixed 2026-08-06 in
> `simulation.py`: the budget is set once per row and each segment gets only what
> remains.
>
> The value is per machine — see section 3. A host far slower than the one the
> global budget was calibrated on will otherwise starve *every* row.


In [4]:
ops.clear_stop()
LOG = RESULTS / f"worker_machine{me['index']}.log"
PIDF = RESULTS / f"worker_machine{me['index']}.pid"
cmd = [PY, '-u', str(HERE / 'run_ensemble.py'), '--design', str(DESIGN),
       '--machine', 'auto', *GRID, '--timeout-s', str(TIMEOUT_S)]
print(' '.join(cmd), '\n')

# SURVIVING THE KERNEL.  The markdown above promises the notebook can be closed.
# On POSIX, start_new_session=True (setsid) delivers exactly that.
#
# On Windows it delivered nothing: start_new_session is POSIX-only and is
# silently ignored there, so the run was an ordinary child of the kernel.
# Neither DETACHED_PROCESS nor CREATE_BREAKAWAY_FROM_JOB fixes it either --
# a kernel is killed with `taskkill /F /T`, and /T walks the PARENT-CHILD
# table, so every descendant goes down with it whatever flags it was born
# with.  The only way past a parent-child walk is to not be a descendant, so
# on Windows the run is spawned through a throwaway launcher that exits
# immediately and orphans it.
# Measured 2026-08-07: without this, a launch from the notebook died the
# instant the kernel shut down -- silently, after printing a healthy
# provenance line.
if sys.platform == 'win32':
    boot = ("import subprocess,sys,json;a=json.loads(sys.argv[1]);"
            "p=subprocess.Popen(a['cmd'],cwd=a['cwd'],"
            "stdout=open(a['log'],'w'),stderr=subprocess.STDOUT,"
            "creationflags=subprocess.DETACHED_PROCESS"
            "|subprocess.CREATE_NEW_PROCESS_GROUP);"
            "open(a['pid'],'w').write(str(p.pid))")
    if PIDF.exists():
        PIDF.unlink()
    subprocess.run([PY, '-c', boot,
                    json.dumps({'cmd': cmd, 'cwd': str(HERE),
                                'log': str(LOG), 'pid': str(PIDF)})],
                   creationflags=subprocess.DETACHED_PROCESS, check=True)
    for _ in range(50):                      # launcher exits as soon as it spawns
        if PIDF.exists():
            break
        time.sleep(0.2)
    pid = PIDF.read_text().strip() if PIDF.exists() else '?'
    print(f'launched pid {pid} -> {LOG.name}   (orphaned; survives the kernel)')
else:
    with open(LOG, 'w') as fh:
        proc = subprocess.Popen(cmd, cwd=HERE, stdout=fh, stderr=subprocess.STDOUT,
                                start_new_session=True)
    print(f'launched pid {proc.pid} -> {LOG.name}   (new session)')

time.sleep(40); print(open(LOG).read()[:1200])


  no STOP flag set.
D:\Repos\RadCluster\.EuroferVenv\Scripts\python.exe -u D:\Repos\RadCluster\RadCluster_2_1\digital_twin\run_ensemble.py --design D:\Repos\RadCluster\RadCluster_2_1\digital_twin\design\T3_rev6.csv --machine auto --equations bin_moment --I 10000 --V 5000 --i-discrete 40 --v-discrete 5 --i-bin 20 --v-bin 20 --shape-function linear --i-mobile-default 40 --v-mobile-default 5 --dose 1.0 --rtol 1e-06 --solver-mode active_window --timeout-s 50400.0 



launched pid 8004 -> worker_machine1.log   (orphaned; survives the kernel)


  detected machine 1 = Matrix-PC (DECLARED speed 0.4), 20 worker(s)
  weights from machines.json (sha 1404a829f1c70a37): 4 participants
machine 1/4  rows 55 (0 done, 55 to run)  workers 20
  provenance {"git_sha": "479d529a4bdec95289675f0f04d50d04c7b6a4f1", "machine_id": "MATRIX-PC2", "solver_sha256": "6d878b0180b264d2", "workbook_sha256": "9253e7a0370af966", "design_sha256": "3461603bdf7818e3", "run_cfg_sha": "482d59556632dff8", "timeout_s": 50400.0, "stop_after_s": 0.0, "workers": 20, "weights_sha": "1404a829f1c70a37", "weights_map_sha": "87969710596d7511", "of": 4, "equations": "bin_moment", "python": "3.13.12"}



## 6 — Monitor


In [ ]:
ops.watch(DESIGN, RESULTS, n_machines=reg['of'], interval=120)   # Ctrl-C to stop watching


  refreshed 06:56:39  (every 120s; interrupt the kernel to stop watching)

CAMPAIGN  T3_rev6.csv   p=19 N=16 conditions=N2,N5,I1
  progress  [######........................................]  13.4%  135/1008 rows
            admissible 135   inadmissible 0   failed 0   missing 873

  timing    per row  mean 2h04m   median 1h35m   p90 3h36m
            core-hours used 279.7   remaining ~1808.9

   machine                 id  assigned   done   left      ETA @6w
         0     Mac.san.rr.com       252     31    221       76h19m
         1     Mac.san.rr.com       252     33    219       75h37m
         2     Mac.san.rr.com       252     36    216       74h35m
         3     Mac.san.rr.com       252     35    217       74h56m

  *** PROVENANCE SPLIT on git_sha - results are NOT comparable:
        1d652999fae45a3e  <- ['MacBook-Pro.local']
        f704409db6cc94f3  <- ['MacBook-Pro.local']
        303ee6121900a50f  <- ['Nasr-Workstation']
        739a160c4ab0e757  <- ['Mac.san.rr.com']

  *

In [ ]:
st = ops.campaign_status(DESIGN, RESULTS, n_machines=reg['of'])
ops.render_status(st, ops.load_targets())


### 6b — Throughput and dose-ladder coverage

Throughput, not wall-time-per-row, is the measure that survives concurrency —
planning from per-row wall measured at low concurrency is what produced a
3780 s estimate against a 17 828 s reality.


In [ ]:
recs = ops.load_results(RESULTS)
walls = [r['wall_s'] for r in recs.values() if r.get('wall_s') and not r.get('solver_rc')]
if walls:
    import statistics
    mean = statistics.mean(walls)
    cap  = sum(float(w) for w in reg['weights'].split(','))
    print(f'rows done {len(walls)}   mean wall {mean:.0f} s   '
          f"throughput {me['slots']*3600/mean:.2f} rows/h on this machine")
    print(f'projected campaign: {len(rows)*mean/cap/3600:.0f} h over {cap:.1f} slot-equivalents')
cov = collections.Counter()
for r in recs.values():
    for rung in (r.get('at_dose') or {}): cov[rung] += 1
for k in sorted(cov, key=float): print(f'  {k:>4s} dpa : {cov[k]:4d} rows')


In [ ]:
# Cross-machine tally: coverage, poolability and REAL throughput.
# Not slots*3600/mean_wall -- the cell above -- because that divides by the wall
# of rows that have LANDED, so while the slow rows are still running it reads
# high (29.3 rows/h against a true 6.8 when this campaign started).
# It also answers the two questions that cell cannot: whether every reported row
# carries the same git/solver/workbook/design/run_cfg hashes (poolability), and
# how many rows each machine still OWES under the frozen weights.
r = subprocess.run([PY, str(HERE / 'campaign_tally.py')],
                   cwd=HERE, capture_output=True, text=True)
print(r.stdout or r.stderr)


## 7 — Graceful stop / resume

Rows in flight finish and are written; no new rows start.


In [ ]:
ops.request_stop('put the real reason here')


In [ ]:
ops.clear_stop()


### 7b — Sync with the other machines

`git` is the transport. `--sync` sends **this** machine's results, pulls
everyone else's, then tallies all four in one step — which is the only way the
coverage and poolability numbers mean anything, since a machine that has not
pushed is indistinguishable from one that has not started.

Add `--no-push` to pull and commit without publishing.

> On Windows, run this **only while no campaign is running on this host**, or
> after a graceful stop. A running worker holds `results/worker_machine*.log`
> open, and git cannot replace an open file — a rebase aborts part-way and
> leaves the working tree half-reverted.


In [ ]:
# Sends this machine's results, pulls the rest, then tallies all four.
r = subprocess.run([PY, str(HERE / 'campaign_tally.py'), '--sync'],
                   cwd=HERE, capture_output=True, text=True)
print(r.stdout or r.stderr)


## 8 — Pool and report (one machine, after everyone has pushed)

Push `results/*.jsonl` **and** `*.manifest.json` — the manifest is what sizes
the next campaign from measured throughput.

Run it **both ways**. If the parameter *ranking* is unchanged with and without
`--require-converged`, truncation did not buy it anything — the empirical form
of the claim the no-gating policy rests on.


In [ ]:
for extra in ([], ['--require-converged']):
    print('='*70); print('  merge_and_sobol', *extra); print('='*70)
    r = subprocess.run([PY, str(HERE / 'merge_and_sobol.py'),
                        '--design', str(DESIGN), '--results', str(RESULTS),
                        '--at-dose', str(reg['grid']['dose']), *extra],
                       cwd=HERE, capture_output=True, text=True)
    print(r.stdout[-4000:])
